# HITL ActionAgent — 데모 노트북 (턴 기반)

기존 사내 LangGraph 구조(**Router → Supervisor → members → FinalAnswerAgent**)에
**Human-In-The-Loop(HITL)** 가 적용된 `ActionAgent` 를 끼워 넣은 결과를 셀 단위로 시연한다.

- 이 노트북은 `app/` 패키지를 **import 해서 쓴다** (코드 원본은 `.py`, 노트북은 검증/시연용).
- 서버 없이 그래프를 직접 `ainvoke` 하며 HITL 왕복을 눈으로 본다.
- 마지막 부록에서 노트북 안에 API 서버를 띄우는 법 + Streamlit 연결법을 다룬다.

## HITL 메커니즘 — 턴 기반 (interrupt 를 쓰지 않는다)

**모든 사용자 입력은 예외 없이 Router → Supervisor 를 경유한다** — 이 불변식이
제1 요구사항이다. LangGraph `interrupt()` 는 재개 시 멈춘 노드로 직행해 이 불변식을
깨므로 쓰지 않는다. 대신:

1. **질문 = 턴의 정상 종료.** 물을 게 생기면 질문 payload 를 `action.awaiting` 에
   싣고 턴을 끝낸다. (Supervisor 가 awaiting 을 보고 FinalAnswer 없이 END)
2. **답변 = 새 턴.** 답도 신규 질문과 똑같이 `{"messages":[HumanMessage(...)]}` 로
   들어와 Router → Supervisor 를 거쳐 ActionAgent 가 소비한다.
3. **상태의 근거는 오직 `action` 스크래치.** 고아 인터럽트/모델 전환 유실 같은
   인터럽트 생명주기 문제가 계열째 없다.

## 지켜야 하는 규칙
1. 실제 실행(side-effect)은 명시적 승인 뒤 execute 에만.
2. HITL 대기 감지는 스트림 종료 후 `aget_state()` 의 `action.awaiting` 으로.
3. finalize/abandon/restart 에서 스크래치 `{}` 리셋 필수.

## 0. 셋업

사내 LLM 게이트웨이가 붙어 있어야 돌아간다 (목업 LLM 은 없다 — 모든 판단을 실제 모델이 한다).

In [1]:
import os, sys, json, asyncio
from pathlib import Path

# 노트북이 notebooks/ 안에 있어도 프로젝트 루트를 import 경로에 넣는다
ROOT = Path.cwd()
if not (ROOT / "app").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from langchain_core.messages import HumanMessage

import app.config as cfg
from app._builder import build_team_graph
from app.actions.mock_db import MOCK_DB

print("ROOT      :", ROOT)
print("gateway  :", cfg.GATEWAY_BASE_URL)
print("carriers  :", list(MOCK_DB["carriers"]))
print("equipment :", list(MOCK_DB["equipment"]))

ROOT      : /home/user/HITL
carriers  : ['6PDMQ283', '3KWQ7712', '9ZXCV456', '7HITL001', 'TESTCAR0001']
equipment : ['STK101', 'STK102', 'PHT201', 'ETC301', 'CLN501', 'DFF401', 'STOCKER9']


## 1. 그래프 빌드 + 구조 확인

In [2]:
graph, checkpointer = build_team_graph()

def C(thread_id):
    """채팅 세션 하나 = thread_id 하나 (대화 맥락 유지)"""
    return {"configurable": {"thread_id": thread_id}}

print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	Router(Router)
	GeneralAgent(GeneralAgent)
	Supervisor(Supervisor)
	StatusAgent(StatusAgent)
	LocationAgent(LocationAgent)
	LogAgent(LogAgent)
	ExtractAgent(ExtractAgent)
	ActionAgent(ActionAgent)
	FinalAnswerAgent(FinalAnswerAgent)
	FinalGeneralAgent(FinalGeneralAgent)
	__end__([<p>__end__</p>]):::last
	ActionAgent --> Supervisor;
	ExtractAgent --> Supervisor;
	GeneralAgent -. &nbsp;FINISH&nbsp; .-> FinalGeneralAgent;
	GeneralAgent -.-> Supervisor;
	LocationAgent --> Supervisor;
	LogAgent --> Supervisor;
	Router -.-> GeneralAgent;
	Router -.-> Supervisor;
	StatusAgent --> Supervisor;
	Supervisor -.-> ActionAgent;
	Supervisor -.-> ExtractAgent;
	Supervisor -. &nbsp;FINISH&nbsp; .-> FinalAnswerAgent;
	Supervisor -.-> LocationAgent;
	Supervisor -.-> LogAgent;
	Supervisor -.-> StatusAgent;
	Supervisor -. &nbsp;END&nbsp; .-> __end__;
	__start__ --> Router;
	FinalAnswerAgent --> __end__;
	FinalG

### ActionAgent 내부 (단일 노드)

`ActionAgent` 는 부모에서 보면 Supervisor 밑 member 노드 하나지만, 내부는 결정적 상태 기계다.
`collect_param` 과 `confirm` 두 노드만 `interrupt()` 를 호출하고, **노드당 정확히 1개**만 둔다.

In [3]:
# ActionAgent 는 서브그래프가 아니라 노드 함수 하나다 (턴 기반 단일 노드).
# 내부 논리 흐름은 함수 docstring 과 README 의 'ActionAgent 내부 논리 흐름' 참고.
from app.actions.node import action_node
print(action_node.__doc__)

턴 기반 단일 노드.

    구성: [진입 판정] -> [수집/검증 루프] -> [턴 종료 dict 반환]
    interrupt 를 쓰지 않으므로 재실행(replay) 함정이 없다 — 위에서 아래로
    읽히는 그대로가 실행 순서다.
    


### 편의 함수 — 인터럽트 상태를 예쁘게 보기

In [4]:
def show_awaiting(snap):
    """상태에서 대기 중인 HITL 질문(action.awaiting)을 꺼내 보여준다.

    턴 기반이라 그래프는 이미 끝나 있다(next=()). 멈춘 게 아니라
    질문을 남기고 턴을 닫은 것이다.
    """
    sc = (getattr(snap, "values", None) or {}).get("action") or {}
    v = sc.get("awaiting")
    if not v:
        print("⏹  HITL 대기 없음 (턴 완료)")
        return None
    print("⏸  HITL 대기: type=%s" % v.get("type"))
    if v.get("field"):
        print("    묻는 파라미터: %s" % v["field"])
    print("    현재 params  : %s" % v.get("params"))
    print("    프롬프트     :")
    for ln in str(v.get("prompt", "")).split("\n"):
        print("      " + ln)
    return v

# 구 이름 호환 (아래 셀들이 그대로 돌게)
show_interrupt = show_awaiting


def answer(snap_cfg, text):
    """HITL 질문에 답한다 — 답도 그냥 **새 턴**이다.

    Command(resume=...) 가 아니라 신규 질문과 똑같은 입력으로 들어가
    Router → Supervisor 를 거쳐 ActionAgent 가 소비한다.
    """
    print("\n>>> 사용자 답변: %r\n" % text)
    return graph.ainvoke({"messages": [HumanMessage(text)]}, snap_cfg)


def final_text(result):
    return result["messages"][-1].content

## 2. 시나리오 A — 필수 파라미터 수집 → 승인 → 실행

`transport`(반송요청명령)의 필수 파라미터는 `carrier_id` + `eqp_id`.
질의에 `eqp_id` 가 없으므로 **HITL 로 되묻는다.**

In [5]:
cfg_a = C("nb-A")
await graph.ainvoke({"messages": [HumanMessage("6PDMQ283 반송해줘")]}, cfg_a)
v = show_interrupt(await graph.aget_state(cfg_a))

[USER] 6PDMQ283 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[NODE] Supervisor: ExtractAgent 선행 실행


...

[ACTION param_check] 미충족 -> ask 'eqp_id'


[ACTION ask_param] ⏸ 질문 남기고 턴 종료 field=eqp_id


[NODE] Supervisor entered


[NODE] Supervisor: HITL 질문 발신(collect_param) -> 턴 종료, 사용자 응답 대기


⏸  HITL 대기: type=collect_param
    묻는 파라미터: eqp_id
    현재 params  : {'carrier_id': '6PDMQ283'}
    프롬프트     :
      목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.


사용자가 목적지를 알려주면 → 파라미터 충족 → 검증 통과 → **두 번째 HITL(실행 승인)**

In [6]:
await answer(cfg_a, "STK102 로 보내줘")
v = show_interrupt(await graph.aget_state(cfg_a))


>>> 사용자 답변: 'STK102 로 보내줘'

[USER] STK102 로 보내줘


[NODE] Router entered


[NODE] Router: 진행 중 액션 감지 -> Supervisor 고정


[NODE] Supervisor entered


[NODE] Supervisor: 진행 중 액션 -> ActionAgent


...

[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION ask_confirm] ⏸ 승인 질문 남기고 턴 종료
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


[NODE] Supervisor entered


[NODE] Supervisor: HITL 질문 발신(confirm) -> 턴 종료, 사용자 응답 대기


⏸  HITL 대기: type=confirm
    현재 params  : {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}
    프롬프트     :
      ⚠️ 반송요청명령 실행 확인
      - 캐리어: 6PDMQ283 (현재 위치 STK101)
      - 목적지: STK102
      이 명령을 정말 실행할까요? (승인/거절)


승인하면 그때서야 `execute` 노드가 실제 액션을 수행한다 (side-effect 는 여기서만).

In [7]:
res = await answer(cfg_a, "승인")
print(final_text(res))


>>> 사용자 답변: '승인'

[USER] 승인


[NODE] Router entered


[NODE] Router: 진행 중 액션 감지 -> Supervisor 고정


[NODE] Supervisor entered


[NODE] Supervisor: 진행 중 액션 -> ActionAgent


...

[ACTION finalize] job=TJ-20260727-0001


[NODE] Supervisor entered


[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered


✅ 반송요청명령 실행 완료
- Job ID: TJ-20260727-0001
- 상태: DISPATCHED
- carrier_id: 6PDMQ283
- dest: STK102


## 3. 시나리오 B — 실행 거절

`dest_req`(목적지요청)는 `carrier_id` 만 있으면 되므로 곧장 승인 단계로 간다.
거절하면 **더 이상 집착하지 않고** 깨끗이 종료한다(재시도 루프 없음).

In [8]:
cfg_b = C("nb-B")
await graph.ainvoke({"messages": [HumanMessage("9ZXCV456 목적지 요청해줘")]}, cfg_b)
show_interrupt(await graph.aget_state(cfg_b))
res = await answer(cfg_b, "아니 하지마")
print(final_text(res))

[USER] 9ZXCV456 목적지 요청해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[NODE] Supervisor: ExtractAgent 선행 실행


...

[ACTION abandon] 사용자가 실행을 거절해 명령을 종료합니다.


[NODE] Supervisor entered


[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered


🚫 목적지요청 을(를) 실행하지 않았습니다.
- 사유: 사용자가 실행을 거절해 명령을 종료합니다.


## 4. 시나리오 C — 유효성 검증 실패 → 잘못된 파라미터만 재수집

`DFF401` 은 목업 DB 에서 **offline** 장비다. validation 이 실패하면
문제가 된 필드만 비우고 수집 루프로 되돌아간다(수렴 보장).

In [9]:
cfg_c = C("nb-C")
await graph.ainvoke({"messages": [HumanMessage("6PDMQ283 를 DFF401 로 반송")]}, cfg_c)
show_interrupt(await graph.aget_state(cfg_c))   # 프롬프트에 '검증 실패' 사유가 실린다

[USER] 6PDMQ283 를 DFF401 로 반송


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[NODE] Supervisor: ExtractAgent 선행 실행


...

[ACTION ask_param] ⏸ 질문 남기고 턴 종료 field=eqp_id


[NODE] Supervisor entered


[NODE] Supervisor: HITL 질문 발신(collect_param) -> 턴 종료, 사용자 응답 대기


⏸  HITL 대기: type=collect_param
    묻는 파라미터: eqp_id
    현재 params  : {'carrier_id': '6PDMQ283'}
    프롬프트     :
      검증 실패: 장비 DFF401 는 현재 오프라인이라 목적지로 지정할 수 없습니다.
      목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.


{'type': 'collect_param',
 'action': 'transport',
 'field': 'eqp_id',
 'prompt': "검증 실패: 장비 DFF401 는 현재 오프라인이라 목적지로 지정할 수 없습니다.\n목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.",
 'params': {'carrier_id': '6PDMQ283'},
 'missing': ['eqp_id']}

In [10]:
await answer(cfg_c, "PHT201")
show_interrupt(await graph.aget_state(cfg_c))
res = await answer(cfg_c, "ㄱㄱ")               # 자연어 승인도 인식
print(final_text(res))


>>> 사용자 답변: 'PHT201'

[USER] PHT201


[NODE] Router entered


[NODE] Router: 진행 중 액션 감지 -> Supervisor 고정


[NODE] Supervisor entered


[NODE] Supervisor: 진행 중 액션 -> ActionAgent


...

[ACTION finalize] job=TJ-20260727-0002


[NODE] Supervisor entered


[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered


✅ 반송요청명령 실행 완료
- Job ID: TJ-20260727-0002
- 상태: DISPATCHED
- carrier_id: 6PDMQ283
- dest: PHT201


## 5. 시나리오 D — 액션 도중 탈출(취소)

모든 interrupt 지점에서 빠져나올 수 있다. resume 답변을 값/승인으로 해석하기 **전에**
취소 의도를 먼저 검사한다.

In [11]:
cfg_d = C("nb-D")
await graph.ainvoke({"messages": [HumanMessage("7HITL001 반송 부탁해")]}, cfg_d)
res = await answer(cfg_d, "아 그냥 취소해줘")
print(final_text(res))

[USER] 7HITL001 반송 부탁해


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[NODE] Supervisor: ExtractAgent 선행 실행


...

[ACTION abandon] 사용자 요청으로 명령을 취소했습니다.


[NODE] Supervisor entered


[NODE] Supervisor: member 응답 완료 -> FinalAnswerAgent


[NODE] FinalAnswer entered


🚫 반송요청명령 을(를) 실행하지 않았습니다.
- 사유: 사용자 요청으로 명령을 취소했습니다.


`/chat/stop` 이 쓰는 abort 센티널도 같은 경로로 정리된다.

In [12]:
cfg_d2 = C("nb-D2")
await graph.ainvoke({"messages": [HumanMessage("7HITL001 반송해줘")]}, cfg_d2)

# /chat/stop 이 하는 일: 턴 기반이라 그래프는 이미 끝나 있으므로
# 스크래치만 리셋하면 끝. 다음 질문은 오염 없이 새로 라우팅된다.
await graph.aupdate_state(cfg_d2, {"action": {}})
snap = await graph.aget_state(cfg_d2)
print("action 스크래치:", snap.values.get("action"))
show_awaiting(snap)

[USER] 7HITL001 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[NODE] Supervisor: ExtractAgent 선행 실행


...

[ACTION param_check] 미충족 -> ask 'eqp_id'


[ACTION ask_param] ⏸ 질문 남기고 턴 종료 field=eqp_id


[NODE] Supervisor entered

[NODE] Supervisor: HITL 질문 발신(collect_param) -> 턴 종료, 사용자 응답 대기


action 스크래치: {}
⏹  HITL 대기 없음 (턴 완료)


## 6. 시나리오 E — 동료 에이전트 값 가져오기 (needs-핸드오프) ★

> ActionAgent 를 Router 에 Supervisor 와 동급으로 붙이면 다른 에이전트와 협업이 안 된다.
> 그래서 Supervisor 밑 member 로 두고, **필요한 값은 동료에게 잠깐 양보해서** 받아온다.

`"Aaa 를 Bbb 있는 위치로 반송"` → ActionAgent 가 `eqp_id` 를 스스로 알 수 없다고 판단하면
`action.needs` 를 기록하고 **interrupt 가 아니라 정상 종료**로 Supervisor 에게 양보한다.
Supervisor 가 LocationAgent 로 라우팅 → 결과를 받아 ActionAgent 재진입 → 그 파라미터는 **묻지 않는다**.

In [13]:
cfg_e = C("nb-E")
await graph.ainvoke(
    {"messages": [HumanMessage("6PDMQ283 를 9ZXCV456 있는 위치로 반송해줘")]}, cfg_e)
v = show_interrupt(await graph.aget_state(cfg_e))
print("\n→ eqp_id 를 사용자에게 묻지 않고 LocationAgent 가 채웠다:", v["params"])

[USER] 6PDMQ283 를 9ZXCV456 있는 위치로 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[NODE] Supervisor: ExtractAgent 선행 실행


...

[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION ask_confirm] ⏸ 승인 질문 남기고 턴 종료
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


[NODE] Supervisor entered


[NODE] Supervisor: HITL 질문 발신(confirm) -> 턴 종료, 사용자 응답 대기


⏸  HITL 대기: type=confirm
    현재 params  : {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}
    프롬프트     :
      ⚠️ 반송요청명령 실행 확인
      - 캐리어: 6PDMQ283 (현재 위치 STK101)
      - 목적지: STK102
      이 명령을 정말 실행할까요? (승인/거절)

→ eqp_id 를 사용자에게 묻지 않고 LocationAgent 가 채웠다: {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


### 로그 분석형 — LogAgent 에게 위임

`"로그 분석해서 원인 장비 피해서 …"` 처럼 **해석용 프롬프트가 필요한 요청**은
ActionAgent 가 직접 하지 않는다. "누가 필요한지"만 선언하고 LogAgent 에게 넘긴다.
(각 에이전트의 해석 프롬프트는 그 에이전트에 남는다 = 복잡도 폭발 회피)

In [14]:
cfg_f = C("nb-F")
await graph.ainvoke(
    {"messages": [HumanMessage("로그 분석해서 원인 장비 피해서 6PDMQ283 반송해줘")]}, cfg_f)
v = show_interrupt(await graph.aget_state(cfg_f))
print("\nfacts(에이전트 간 공유 팩트):")
print(json.dumps((await graph.aget_state(cfg_f)).values.get("facts", {}),
                 ensure_ascii=False, indent=2)[:800])

[USER] 로그 분석해서 원인 장비 피해서 6PDMQ283 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[NODE] Supervisor: ExtractAgent 선행 실행


...

[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION ask_confirm] ⏸ 승인 질문 남기고 턴 종료
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


[NODE] Supervisor entered


[NODE] Supervisor: HITL 질문 발신(confirm) -> 턴 종료, 사용자 응답 대기


⏸  HITL 대기: type=confirm
    현재 params  : {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}
    프롬프트     :
      ⚠️ 반송요청명령 실행 확인
      - 캐리어: 6PDMQ283 (현재 위치 STK101)
      - 목적지: STK102
      이 명령을 정말 실행할까요? (승인/거절)

facts(에이전트 간 공유 팩트):
{
  "extracted": {
    "fab": "M16",
    "carrier_ids": [
      "6PDMQ283"
    ],
    "eqp_ids": [],
    "gate_pass": true
  },
  "log_analysis": {
    "carrier_id": "6PDMQ283",
    "combos": [
      {
        "reason": "E-STALL",
        "description": "OHT stall detected",
        "count": 3,
        "eqp": "ETC301",
        "first_t": "08:01:02"
      },
      {
        "reason": "E-PORT",
        "description": "port not ready",
        "count": 2,
        "eqp": "ETC301",
        "first_t": "08:03:40"
      }
    ],
    "cause_eqp": "ETC301",
    "recommended_dest": "STK102"
  }
}


## 7. 시나리오 G — HITL 질문에 **참조형으로** 답하기

파라미터를 물었는데 사용자가 값 대신 `"9ZXCV456 있는 위치로 채워줘"` 라고 답하는 경우.
답변 해석은 4분기다: ①취소 ②리터럴 ③참조/분석형(needs 핸드오프) ④해석불능(재질문).

In [15]:
cfg_g = C("nb-G")
await graph.ainvoke({"messages": [HumanMessage("6PDMQ283 반송해줘")]}, cfg_g)
show_interrupt(await graph.aget_state(cfg_g))
await answer(cfg_g, "9ZXCV456 있는 위치로 채워줘")     # 값이 아니라 참조
v = show_interrupt(await graph.aget_state(cfg_g))
print("\n→ LocationAgent 를 경유해 자동으로 채워짐:", v["params"])

[USER] 6PDMQ283 반송해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[NODE] Supervisor: ExtractAgent 선행 실행


...

[TOOL transport_confirm] guidance rendered (84 chars)


[ACTION ask_confirm] ⏸ 승인 질문 남기고 턴 종료
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)


[NODE] Supervisor entered


[NODE] Supervisor: HITL 질문 발신(confirm) -> 턴 종료, 사용자 응답 대기


⏸  HITL 대기: type=confirm
    현재 params  : {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}
    프롬프트     :
      ⚠️ 반송요청명령 실행 확인
      - 캐리어: 6PDMQ283 (현재 위치 STK101)
      - 목적지: STK102
      이 명령을 정말 실행할까요? (승인/거절)

→ LocationAgent 를 경유해 자동으로 채워짐: {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'}


## 8. 시나리오 H — 액션 자체가 불명확하면 액션을 묻는다

`transport` 인지 `dest_req` 인지 모를 때는 **어떤 명령인지**부터 HITL 로 확인한다.

In [16]:
cfg_h = C("nb-H")
await graph.ainvoke({"messages": [HumanMessage("6PDMQ283 에 명령 실행해줘")]}, cfg_h)
show_interrupt(await graph.aget_state(cfg_h))
await answer(cfg_h, "반송으로")
show_interrupt(await graph.aget_state(cfg_h))

[USER] 6PDMQ283 에 명령 실행해줘


[NODE] Router entered


[AGENT] router(fake) -> supervisor


[NODE] Supervisor entered


[NODE] Supervisor: ExtractAgent 선행 실행


...

[ACTION ask_param] ⏸ 질문 남기고 턴 종료 field=eqp_id


[NODE] Supervisor entered


[NODE] Supervisor: HITL 질문 발신(collect_param) -> 턴 종료, 사용자 응답 대기


⏸  HITL 대기: type=collect_param
    묻는 파라미터: eqp_id
    현재 params  : {'carrier_id': '6PDMQ283'}
    프롬프트     :
      목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.


{'type': 'collect_param',
 'action': 'transport',
 'field': 'eqp_id',
 'prompt': "목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.",
 'params': {'carrier_id': '6PDMQ283'},
 'missing': ['eqp_id']}

## 9. 상태 설계 확인 — 왜 messages 밖에 두는가

`messages` 에는 최근 4턴만 남기는 리듀서가 걸려 있다. HITL 문답을 messages 에만 두면
**진행 중이던 액션이 잘려서 깨진다.** 그래서 액션 제어 상태는 `action`(교체) /
`facts`(병합) 필드에 두고, 리듀서가 건드리지 못하게 한다.

In [17]:
snap = await graph.aget_state(C("nb-H"))
sc = snap.values.get("action", {})
print("action 스크래치 (limiter 영향 없음):")
for k in ("action", "phase", "params", "missing", "pending_field",
          "collect_retries", "validate_retries", "hops"):
    if k in sc:
        print(f"  {k:16} = {sc[k]}")
print("\nmessages 개수:", len(snap.values.get("messages", [])))

action 스크래치 (limiter 영향 없음):
  action           = transport
  phase            = collecting
  params           = {'carrier_id': '6PDMQ283'}
  missing          = ['eqp_id']
  pending_field    = eqp_id
  collect_retries  = 0
  validate_retries = 0
  hops             = 0

messages 개수: 5


## 10. 부록 — 노트북에서 API 서버 띄우고 Streamlit 붙이기

**질문: ipynb 에서 서버 띄우고 Streamlit 연결이 되나?**

- **API 서버**: 된다. 아래 셀처럼 uvicorn 을 백그라운드 스레드로 띄우면 노트북을 계속 쓰면서
  `http://localhost:8000` 으로 호출할 수 있다.
- **Streamlit**: 노트북 셀 안에서는 못 띄운다. Streamlit 은 자체 프로세스로 떠야 하므로
  **터미널에서** `streamlit run streamlit_app.py` 로 실행하고, 그 UI 가 위 API 를 호출하게 한다.

정리하면 권장 실행 형태는 이렇다.

```bash
# 터미널 1
uvicorn app.api.main:app --reload --port 8000
# 터미널 2
streamlit run streamlit_app.py
```

In [18]:
# 노트북 안에서 API 서버 백그라운드 기동 (선택)
import threading, time
import uvicorn
from app.api.main import app as fastapi_app

def _serve():
    uvicorn.run(fastapi_app, host="127.0.0.1", port=8000, log_level="warning")

if not any(t.name == "uvicorn-nb" for t in threading.enumerate()):
    threading.Thread(target=_serve, name="uvicorn-nb", daemon=True).start()
    time.sleep(3)

import httpx
print(httpx.get("http://127.0.0.1:8000/llm/api/health", timeout=10).json())

[MCP] connect (스텁 — 사내에서 실제 서버 연결로 교체)

{'ok': True, 'default_model': 'GaiA-LLM-Latest', 'cached_graphs': [], 'time': '2026-07-28T08:45:41.046569'}


### 서버에 SSE 로 질의해 보기 (Streamlit 이 하는 일과 동일)

In [19]:
import httpx, json

# 백엔드는 최종 답변을 raw text 로 흘리고, 제어 정보만 \x1e 로 시작하는
# JSON 한 줄로 보낸다. 아래에서 그 둘을 갈라낸다.
EVENT_PREFIX = "\x1e"


def ask(thread_id, query, model_name=None, recursion_limit=20):
    """/chat/stream 을 호출하고 진행 상황을 요약 출력. 반환값은 needs_input payload."""
    payload = {"query": query, "thread_id": thread_id,
               "model_name": model_name, "recursion_limit": recursion_limit}

    answer, needs, usage, buffer = "", None, None, ""

    def handle(ev):
        """제어 프레임 하나 처리."""
        nonlocal needs, usage
        t = ev.get("type")
        if t == "node_enter":
            print(f"  ▶️  {ev['agent']}")
        elif t == "tool_call":
            print(f"  🔧 {ev.get('tool')} {ev.get('result') or ''}")
        elif t == "agent_status":
            print(f"  💬 {ev.get('agent')}: {ev.get('detail')}")
        elif t == "needs_input":
            needs = ev
        elif t == "usage":
            usage = ev

    with httpx.Client(timeout=None) as c:
        with c.stream("POST", "http://127.0.0.1:8000/llm/api/chat/stream",
                      json=payload) as r:
            for raw in r.iter_text():
                buffer += raw
                while EVENT_PREFIX in buffer:
                    head, _, rest = buffer.partition(EVENT_PREFIX)
                    if head:
                        answer += head
                    if "\n" not in rest:
                        buffer = EVENT_PREFIX + rest
                        break
                    line, _, remainder = rest.partition("\n")
                    handle(json.loads(line))
                    buffer = remainder
                else:
                    answer += buffer
                    buffer = ""

    if needs:
        print("\n⏸ HITL:", needs.get("kind"))
        print(needs.get("prompt"))
    if answer.strip():
        print("\n💬", answer)
    if usage:
        print(f"\n🔢 {usage['total_tokens']} tok · ⏱ TTFT {usage['ttft_ms']}ms / "
              f"총 {usage['elapsed_ms']}ms (사람대기 {usage['human_wait_ms']}ms 제외 "
              f"{usage['compute_ms']}ms) · HITL {usage['hitl_rounds']}회")
    return needs


ask("nb-api", "6PDMQ283 반송해줘")

[GRAPH] 빌드 (model=default)


[STREAM] thread=nb-api NEW TURN <- '6PDMQ283 반송해줘'


[USER] 6PDMQ283 반송해줘  ▶️  Router



[NODE] Router entered


[AGENT] router(fake) -> supervisor


...

[NODE] Supervisor entered  ▶️  Supervisor



[NODE] Supervisor: HITL 질문 발신(collect_param) -> 턴 종료, 사용자 응답 대기


  💬 Supervisor: 사용자 응답 대기 — 턴 종료
[STREAM] thread=nb-api ⏸ HITL 대기: collect_param



⏸ HITL: collect_param
목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.

🔢 146 tok · ⏱ TTFT Nonems / 총 53ms (사람대기 0ms 제외 53ms) · HITL 0회


{'type': 'needs_input',
 'kind': 'collect_param',
 'step_history': ['Router',
  'Supervisor',
  'ExtractAgent',
  'Supervisor',
  'ActionAgent',
  'Supervisor'],
 'action': 'transport',
 'field': 'eqp_id',
 'prompt': "목적지 장비 ID를 알려주세요. (예: STK102) 다른 캐리어가 있는 위치로 보내려면 '<캐리어ID> 위치로'라고 답하셔도 됩니다.",
 'params': {'carrier_id': '6PDMQ283'},
 'missing': ['eqp_id']}

In [20]:
ask("nb-api", "STK102")

[STREAM] thread=nb-api HITL 답변(collect_param) <- 'STK102'

  💬 HITL: 사용자 응답 수신(collect_param) — Router/Supervisor 경유 재개
[USER] STK102
  ▶️  Router


[NODE] Router entered


[NODE] Router: 진행 중 액션 감지 -> Supervisor 고정


  💬 Router: 의도 분류 중
[NODE] Supervisor entered
  ▶️  Supervisor


...

  ▶️  Supervisor[NODE] Supervisor entered



[NODE] Supervisor: HITL 질문 발신(confirm) -> 턴 종료, 사용자 응답 대기


  💬 Supervisor: 사용자 응답 대기 — 턴 종료
[STREAM] thread=nb-api ⏸ HITL 대기: confirm



⏸ HITL: confirm
⚠️ 반송요청명령 실행 확인
- 캐리어: 6PDMQ283 (현재 위치 STK101)
- 목적지: STK102
이 명령을 정말 실행할까요? (승인/거절)

🔢 146 tok · ⏱ TTFT Nonems / 총 148ms (사람대기 64ms 제외 84ms) · HITL 1회


{'type': 'needs_input',
 'kind': 'confirm',
 'step_history': ['Router', 'Supervisor', 'ActionAgent', 'Supervisor'],
 'action': 'transport',
 'prompt': '⚠️ 반송요청명령 실행 확인\n- 캐리어: 6PDMQ283 (현재 위치 STK101)\n- 목적지: STK102\n이 명령을 정말 실행할까요? (승인/거절)',
 'params': {'carrier_id': '6PDMQ283', 'eqp_id': 'STK102'},
 'options': ['승인', '거절']}

In [21]:
ask("nb-api", "승인")

[STREAM] thread=nb-api HITL 답변(confirm) <- '승인'

  💬 HITL: 사용자 응답 수신(confirm) — Router/Supervisor 경유 재개[USER] 승인


[NODE] Router entered

  ▶️  Router


[NODE] Router: 진행 중 액션 감지 -> Supervisor 고정


  💬 Router: 의도 분류 중


...

 6PDMQ28

3
- dest

: STK102

[saved] /home/user/HITL/logs/devLogs/2026-07/2026-07-28.jsonl



💬 ✅ 반송요청명령 실행 완료
- Job ID: TJ-20260727-0003
- 상태: DISPATCHED
- carrier_id: 6PDMQ283
- dest: STK102

🔢 396 tok · ⏱ TTFT 266ms / 총 283ms (사람대기 153ms 제외 130ms) · HITL 2회


## 11. 로그 확인

`logs/{env}/{YYYY-MM}/{YYYY-MM-DD}.jsonl` 에 턴당 1건이 기록된다.
HITL 로 여러 번의 `/chat/stream` 에 걸친 액션도 **완료 시점에 1건**으로 합산된다.

In [22]:
from app.api.routes import _get_log_dir, kst_date_str, read_log_records
p = Path(_get_log_dir()) / f"{kst_date_str()}.jsonl"
print(p)
if p.exists():
    recs = read_log_records(p)
    print(f"오늘 기록된 턴: {len(recs)}건\n")
    rec = recs[-1]
    for k in ("thread_id", "query", "route", "outcome"):
        print(f"{k:12}: {rec.get(k)}")
    print("step_history:", [s["node"] for s in rec["step_history"]])
    print("token_cost  :", json.dumps(rec["token_cost"], ensure_ascii=False))
    print("time_cost   :", json.dumps(rec["time_cost"], ensure_ascii=False))
    print("hitl        :", rec["hitl"])
else:
    print("아직 로그 없음 — 위 API 셀을 먼저 실행하세요.")

/home/user/HITL/logs/devLogs/2026-07/2026-07-28.jsonl
오늘 기록된 턴: 1건

thread_id   : nb-api
query       : 6PDMQ283 반송해줘
route       : supervisor
outcome     : complete
step_history: ['Router', 'Supervisor', 'ExtractAgent', 'Supervisor', 'ActionAgent', 'Supervisor', 'Router', 'Supervisor', 'ActionAgent', 'Supervisor', 'Router', 'Supervisor', 'ActionAgent', 'Supervisor', 'FinalAnswerAgent']
token_cost  : {"user_input_tokens": 4, "per_agent": {"Router": {"input": 14, "output": 7, "calls": 1}, "ExtractAgent": {"input": 30, "output": 23, "calls": 1}, "Supervisor": {"input": 16, "output": 7, "calls": 1}, "ActionAgent": {"input": 29, "output": 20, "calls": 1}, "FinalAnswerAgent": {"input": 218, "output": 32, "calls": 1}}, "agent_input_tokens": 307, "agent_output_tokens": 89, "total_tokens": 396}
time_cost   : {"ttft_ms": 266, "elapsed_ms": 284, "human_wait_ms": 153, "compute_ms": 131}
hitl        : {'rounds': 2, 'stream_calls': 3}
